# Opis projektu i dobór priorów

Ten notebook opisuje aktualną wersję projektu: **modele drużynowe** (Stan), które przewidują punkty **jednej drużyny** w sezonie. Złożenie pełnej tabeli ligowej z wielu takich predykcji jest pokazane osobno w notebooku **05** (`05_backtest_models_comparison`).

W projekcie używamy dwóch modeli drużynowych:

- `stan/team_static.stan` - model statyczny z jednym globalnym skillem drużyny,
- `stan/team_hierarchical.stan` - model hierarchiczny z trwałą jakością drużyny i wspólnym efektem sezonu.

Oba modele mają **wejście = jedna drużyna w sezonie** i **wyjście = rozkład punktów** tej drużyny. Model 1 używa cech procesowych oraz indeksu klubu, natomiast Model 2 używa indeksu klubu, indeksu sezonu i statusu beniaminka.

### Notebooki w projekcie

| Plik | Treść |
|------|--------|
| `00_project_overview_and_priors` | Ten notebook — opis projektu i priory |
| `01_eda_and_tables` | EDA i budowa tabel sezonowych |
| `02_model1_static_team` | Model 1 (static), fit, PPC, backtest |
| `03_model2_hierarchical_team` | Model 2 (hierarchical), fit, PPC, backtest |
| `04_forecast_2627_comparison` | Forecast 2026/27, LOO/WAIC, porównanie modeli |
| `05_backtest_models_comparison` | Backtest 2025/26 — Model 1 vs Model 2 |

In [6]:
import numpy as np
import pandas as pd

from helping_functions import (
    load_matches,
    load_season_tables,
    prepare_table_stan_static,
    prepare_table_stan_hierarchical,
    BACKTEST_TRAIN_SEASONS,
    BACKTEST_TEST_SEASON,
    FORECAST_TRAIN_SEASONS,
    FORECAST_SEASON_LABEL,
    STUDENT_T_NU,
)

matches = load_matches()
tables = load_season_tables(matches, BACKTEST_TRAIN_SEASONS)

print(f"Mecze w danych: {len(matches)}")
print(f"Sezony w danych: {sorted(matches['season'].unique())}")
print(f"Sezony treningowe backtestu: {BACKTEST_TRAIN_SEASONS[0]}-{BACKTEST_TRAIN_SEASONS[-1]}")
print(f"Sezon testowy backtestu: {BACKTEST_TEST_SEASON}")
print(f"Sezony treningowe finalnego forecastu: {FORECAST_TRAIN_SEASONS[0]}-{FORECAST_TRAIN_SEASONS[-1]}")
print(f"Forecast sezonu: {FORECAST_SEASON_LABEL}")
print(f"Student-t nu w modelach drużynowych: {STUDENT_T_NU} i 2.0 dla hierarchical")

Mecze w danych: 6460
Sezony w danych: ['0910', '1011', '1112', '1213', '1314', '1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425', '2526']
Sezony treningowe backtestu: 0910-2425
Sezon testowy backtestu: 2526
Sezony treningowe finalnego forecastu: 0910-2526
Forecast sezonu: 2627
Student-t nu w modelach drużynowych: 5.0 i 2.0 dla hierarchical


## Cel projektu

Celem jest probabilistyczne przewidywanie **liczby punktów drużyny** w sezonie Premier League. Modele nie przewidują pojedynczych wyników meczów ani całej tabeli naraz — każda predykcja dotyczy **jednego klubu**.

Przepływ jest następujący:

1. Wczytujemy historyczne mecze.
2. Z meczów budujemy końcowe tabele sezonów (dane treningowe).
3. Dla każdej pary drużyna–sezon wyliczamy cechy opisujące jakość i formę.
4. Trenujemy model Stan na punktach końcowych (jeden wiersz = jedna drużyna w sezonie).
5. Dla prognozy podajemy cechy **jednej drużyny** i otrzymujemy rozkład jej punktów (`predict_team_points`).
6. *(Opcjonalnie, `05_backtest_models_comparison`)* Łączymy predykcje 20 klubów w tabelę ligową przez sortowanie po punktach.

Wynik główny modelu: rozkład punktów dla danej drużyny (mediana, kwantyle, niepewność).

## Dane wejściowe

Źródłem danych są pliki `season-*.csv` z katalogu `Data/datahub.io`. Każdy wiersz to jeden mecz. Do modeli drużynowych używamy danych po agregacji do poziomu drużyna–sezon.

Najpierw funkcja `compute_table` buduje końcową tabelę sezonu. Następnie `load_season_tables(..., with_features=True)` dokleja cechy procesowe. Domyślnie używane są **lagowane cechy**, czyli dla targetu z sezonu `s` model dostaje informacje z sezonu `s-1`.

Jedna obserwacja w modelu oznacza:

```text
jedna drużyna w jednym sezonie -> końcowe punkty + cechy
```

Dla 16 sezonów treningowych po 20 drużyn mamy 320 obserwacji. Lagowanie cech jest ważne, bo zapobiega przeciekowi informacji: model nie widzi strzałów celnych ani formy z sezonu, którego punkty ma przewidzieć.

In [7]:
tables[["season", "team", "Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].head(10)

,season,team,Pts,sot_diff_pg,pts_lag1,ppg_last10
0,0910,Chelsea,86,0.0,0.0,0.0
1,0910,Man United,85,0.0,0.0,0.0
2,0910,Arsenal,75,0.0,0.0,0.0
3,0910,Tottenham,70,0.0,0.0,0.0
4,0910,Man City,67,0.0,0.0,0.0
5,0910,Aston Villa,64,0.0,0.0,0.0
6,0910,Liverpool,63,0.0,0.0,0.0
7,0910,Everton,61,0.0,0.0,0.0
8,0910,Birmingham,50,0.0,0.0,0.0
9,0910,Blackburn,50,0.0,0.0,0.0


## Cechy używane w modelach

Zmienna objaśniana:

- `Pts` - końcowa liczba punktów drużyny w sezonie.

Predyktory dostępne w tabelach:

- `sot_diff_pg` - różnica strzałów celnych na mecz z poprzedniego sezonu,
- `pts_lag1` - punkty z poprzedniego sezonu; dla beniaminków i pierwszego sezonu w danych przyjmujemy 0,
- `ppg_last10` - punkty na mecz w ostatnich 10 kolejkach poprzedniego sezonu,
- `is_promoted` - 1, jeśli drużyna nie grała w poprzednim sezonie Premier League (beniaminek), inaczej 0.

Model 1 używa trzech cech procesowych (`sot_diff_pg`, `pts_lag1`, `ppg_last10`) oraz `is_promoted`. Model 2 celowo nie używa cech procesowych w likelihoodzie; wykorzystuje tylko `team`, `season` i `is_promoted`.

Wszystkie trzy cechy ciągłe są standaryzowane do z-score w funkcji `_standardize_features`. Dzięki temu współczynniki regresji w Modelu 1 mówią o zmianie oczekiwanych punktów przy zmianie cechy o jedno odchylenie standardowe. Standaryzacja jest liczona tylko na danych treningowych, a potem te same średnie i odchylenia są używane dla forecastu Modelu 1.

In [8]:
tables[["Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].describe().round(2)

,Pts,sot_diff_pg,pts_lag1,ppg_last10
count,320.00,320.00,320.00,320.00
mean,52.42,0.30,45.02,1.21
std,17.70,1.70,26.80,0.78
min,12.00,-3.76,0.00,0.00
25%,40.00,-0.74,38.00,0.70
50%,48.00,0.00,47.00,1.30
75%,65.25,1.14,63.25,1.80
max,100.00,6.03,100.00,3.00


## Dlaczego Student-t?

Oba modele drużynowe używają rozkładu Studenta:

$$Pts_n \sim t_{\nu}(\mu_n, \sigma_{pts})$$

W projekcie `nu = 5` jest stałe (2.0 dla hierachical). Student-t jest bardziej odporny na nietypowe sezony niż rozkład normalny. To ma sens dla tabel ligowych, bo zdarzają się sezony odstające: bardzo mocny mistrz, wyjątkowo słaby spadkowicz, nietypowy sezon beniaminka albo drużyna z dużymi problemami kadrowymi.

`sigma_pts` nie oznacza surowego odchylenia standardowego punktów w tabeli. Oznacza skalę reszt po uwzględnieniu cech i skillu.

## Model 1: `team_static.stan`

Model statyczny zakłada, że każda drużyna ma jeden latentny skill w całym okresie treningowym.

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + \beta_{pts} skill_t + \beta_{sot} x^{sot}_{s,t} + \beta_{lag} x^{lag}_{s,t} + \beta_{form} x^{form}_{s,t} + \beta_{promoted}\, is\_promoted_{s,t}$$

Do Stana wrzucamy:

- `N` - liczba obserwacji drużyna-sezon,
- `T` - liczba drużyn obecnych w treningu,
- `team[n]` - indeks drużyny,
- `pts[n]` - końcowe punkty,
- `sot_diff_pg[n]`, `pts_lag1[n]`, `ppg_last10[n]` - wystandaryzowane cechy,
- `is_promoted[n]` - 1 dla beniaminka / nowej drużyny w sezonie, inaczej 0,
- `nu` - stopnie swobody rozkładu Studenta.

Mapowanie cech: $x^{sot}$ = `sot_diff_pg`, $x^{lag}$ = `pts_lag1`, $x^{form}$ = `ppg_last10`.

Ten model jest prosty i stabilny. Dobrze działa, jeśli stała jakość drużyny jest silnym sygnałem, a różnice między sezonami da się częściowo opisać cechami.

In [9]:
static_data, static_team_to_idx, static_teams, static_feature_stats = prepare_table_stan_static(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "T", "nu"} else v) for k, v in static_data.items()}

{'N': 320,
 'T': 42,
 'nu': 5.0,
 'team': 320,
 'pts': 320,
 'sot_diff_pg': 320,
 'pts_lag1': 320,
 'ppg_last10': 320,
 'is_promoted': 320}

### Priory w `team_static.stan`

```stan
intercept ~ normal(52, 15);
beta_pts ~ normal(10, 10);       // parametr ograniczony do beta_pts >= 0
beta_sot ~ normal(0, 10);
beta_lag ~ normal(0, 1);
beta_form ~ normal(0, 10);
beta_promoted ~ normal(-8, 8);
log_sigma_pts ~ normal(log(15), 0.6);
skill ~ std_normal();
```

Priory zostały dobrane na **skali punktów** i na podstawie wiedzy dziedzinowej o lidze. Drużyna Premier League gra 38 meczów, więc sezonowy dorobek punktowy naturalnie mieści się mniej więcej na skali 0-100 punktów. Priory są słabo informatywne: ograniczają absurdalne skale, ale zostawiają dużo miejsca, żeby posterior został określony przez dane.

`intercept ~ normal(52, 15)` to bazowy poziom punktów dla przeciętnej drużyny, gdy wystandaryzowane cechy są blisko 0, `skill = 0` i `is_promoted = 0`. Środek 52 traktujemy jako szeroki poziom środka tabeli na skali Premier League, nie jako estymatę z danych. Odchylenie 15 daje orientacyjny 95% zakres `52 ± 30`, czyli około 22-82 punkty, więc baseline dopuszcza zarówno słabszy, jak i mocniejszy sezon przeciętnej drużyny.

`skill ~ std_normal()` nadaje latentnej jakości drużyny standardową skalę. Ponieważ `skill` jest typu `sum_to_zero_vector[T]`, średni skill drużyn jest równy zero. Dzięki temu `intercept` oznacza poziom przeciętnej drużyny, a intercept i efekty drużyn nie mogą się swobodnie przesuwać względem siebie.

`beta_pts ~ normal(10, 10)` przelicza jedną jednostkę latentnego `skill` na punkty. W Stan ten parametr ma ograniczenie `lower=0`, więc prior jest w praktyce szerokim dodatnim priorem. Środek 10 mówi, że różnica jednej jednostki skilla może odpowiadać około 10 punktom, ale duże odchylenie pozwala zarówno na znacznie mniejszy, jak i większy wpływ. Ograniczenie dodatnie ustala orientację: większy `skill` oznacza więcej oczekiwanych punktów.

`beta_sot ~ normal(0, 10)` i `beta_form ~ normal(0, 10)` dotyczą wystandaryzowanych cech. Zmiana cechy o jedno odchylenie standardowe może rozsądnie zmienić wynik punktowy o kilka lub kilkanaście punktów, ale prior jest wycentrowany w 0, bo przed dopasowaniem nie wymuszamy konkretnej siły efektu. Orientacyjny 95% zakres `-20` do `+20` punktów jest celowo szeroki.

`beta_lag ~ normal(0, 1)` jest bardziej regularyzujący. Punkty z poprzedniego sezonu mogą pomagać, ale Model 1 ma już latentny skill drużyny, więc bardzo duży efekt laga dublowałby ten sam sygnał długookresowej jakości. Ponieważ `pts_lag1` jest standaryzowane, ten prior utrzymuje bezpośredni efekt laga mały, chyba że dane wyraźnie pokażą inaczej.

`beta_promoted ~ normal(-8, 8)` wyraża słabe przekonanie dziedzinowe, że beniaminkowie lub nowe drużyny mogą być słabsze od ustabilizowanych klubów Premier League. Środek `-8` jest założeniem przed danymi, nie wartością dopasowaną. Orientacyjny 95% zakres to około `-24` do `+8` punktów, więc prior nadal dopuszcza małą karę albo nawet dodatni efekt beniaminka.

`log_sigma_pts ~ normal(log(15), 0.6)` opisuje resztkową niepewność na dodatniej skali przez `sigma_pts = exp(log_sigma_pts)`. Mediana skali reszt to 15 punktów. Bez ograniczeń orientacyjny 95% zakres wynosi `15 * exp(±1.2)`, czyli około 4.5-50 punktów; ograniczenia w Stan obcinają go do `0.5-30` dla stabilności numerycznej. To utrzymuje szum na sensownej piłkarskiej skali bez dopasowywania prioru do obserwowanych danych.


## Model 2: `team_hierarchical.stan`

Model hierarchiczny jest celowo innym modelem niż Model 1: nie używa cech procesowych (`sot_diff_pg`, `pts_lag1`, `ppg_last10`) w likelihoodzie. Opiera predykcję na latentnej jakości drużyny, wspólnym efekcie sezonu i statusie beniaminka:

$$skill_{s,t} = team\_skill_t + season\_effect_s$$

- `team_skill[t]` - trwała jakość drużyny,
- `season_effect[s]` - wspólne odchylenie poziomu punktów w danym sezonie.
- brak osobnego `season_dev[s,t]`; indywidualne niespodzianki drużyna-sezon zostają w residual Student-t,
- brak `sot_diff_pg`, `pts_lag1`, `ppg_last10`; Model 2 testuje, ile daje sama latentna struktura drużyn i sezonów.

Równanie modelu:

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + team\_skill_t + season\_effect_s + \beta_{promoted}\, is\_promoted_{s,t}$$

To jest czysty model hierarchiczny: trwała jakość drużyny jest częściowo poolingowana przez `tau_team`, sezony są częściowo poolingowane przez `tau_season`, a pojedyncze niespodzianki pozostają w residual Student-t.

In [10]:
hier_data, hier_team_to_idx, hier_teams, hier_season_to_idx, hier_feature_stats = prepare_table_stan_hierarchical(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "S", "T", "nu"} else v) for k, v in hier_data.items()}

{'N': 320,
 'S': 16,
 'T': 42,
 'nu': 5.0,
 'season': 320,
 'team': 320,
 'pts': 320,
 'is_promoted': 320}

### Co dodatkowo trafia do modelu hierarchicznego?

Model hierarchiczny dostaje czystszy zestaw danych niż model statyczny:

- `N` - liczba obserwacji drużyna-sezon,
- `S` - liczba sezonów,
- `T` - liczba drużyn,
- `season[n]` - indeks sezonu obserwacji,
- `team[n]` - indeks drużyny obserwacji,
- `pts[n]` - końcowe punkty,
- `is_promoted[n]` - status beniaminka / nowej drużyny,
- `nu` - stopnie swobody rozkładu Studenta.

Model 2 nie dostaje już `sot_diff_pg`, `pts_lag1` ani `ppg_last10` w danych Stana. Efekty hierarchiczne są zapisane jako `sum_to_zero_vector[T] team_skill_z` oraz `sum_to_zero_vector[S] season_effect_z`, a następnie przeskalowane przez `tau_team` i `tau_season`.

### Priory w `team_hierarchical.stan`

```stan
intercept ~ normal(50, 25);
beta_promoted ~ normal(-5, 15);
log_sigma_pts ~ normal(log(15), 0.8);
log_tau_team ~ normal(log(12), 0.8);
log_tau_season ~ normal(log(5), 0.8);
team_skill_z ~ std_normal();
season_effect_z ~ std_normal();
```

Model 2 jest celowo mniej związany z cechami procesowymi niż Model 1, więc jego priory są szersze. Nadal są dobrane przed analizą dopasowania modelu: używamy tylko skali punktów w piłce, struktury 38-meczowego sezonu i jakościowej wiedzy, że różnice między drużynami powinny zwykle być większe niż wspólne przesunięcia między sezonami.

`intercept ~ normal(50, 25)` to oczekiwany poziom punktów dla przeciętnej drużyny w przeciętnym sezonie, bez statusu beniaminka. Środek 50 jest okrągłym baseline'em na skali ligi: leży mniej więcej w środku rozsądnie możliwego zakresu 0-100 punktów. Odchylenie 25 daje orientacyjny 95% zakres `50 ± 50`, czyli około 0-100 punktów. To prior celowo szerszy niż konkretna historyczna średnia.

`beta_promoted ~ normal(-5, 15)` to słaby prior dziedzinowy dla beniaminków lub nowych drużyn. Środek `-5` mówi, że takie drużyny mogą być trochę słabsze, ale odchylenie 15 daje orientacyjny 95% zakres około `-35` do `+25` punktów. Prior nie wymusza więc słabości beniaminka; dopuszcza dużą karę, brak kary, a nawet bardzo dobry sezon.

`team_skill_z ~ std_normal()` nadaje każdej drużynie latentne odchylenie od przeciętnej drużyny. Ponieważ jest to `sum_to_zero_vector[T]`, suma odchyleń drużyn wynosi zero. Efekty drużyn są więc względne: dodatni efekt oznacza drużynę powyżej średniej, ujemny poniżej średniej.

`log_tau_team ~ normal(log(12), 0.8)` kontroluje, jak bardzo skille drużyn mogą różnić się na skali punktów. Ponieważ `team_skill = tau_team * team_skill_z`, `tau_team` jest przelicznikiem długookresowej jakości klubu na punkty. Mediana prioru to 12 punktów. Nieucięty orientacyjny 95% zakres to `12 * exp(±1.6)`, czyli około 2.4-59 punktów, a ograniczenia Stan obcinają go do `0.5-45`. To szeroko dopuszcza duże różnice między czołówką i dołem ligi, ale nie generuje głównie absurdalnych tabel.

`season_effect_z ~ std_normal()` nadaje każdemu sezonowi wspólne odchylenie, działające na wszystkie drużyny w tym sezonie. Ten wektor też sumuje się do zera, więc `intercept` pozostaje baseline'em przeciętnego sezonu, zamiast zostać pochłonięty przez efekty sezonowe.

`log_tau_season ~ normal(log(5), 0.8)` kontroluje wielkość wspólnych przesunięć sezonowych. Mediana to 5 punktów, mniej niż dla `tau_team`, bo wiedza dziedzinowa sugeruje, że różnice między klubami powinny być zwykle większe niż różnice między całymi sezonami. Nieucięty orientacyjny 95% zakres to `5 * exp(±1.6)`, czyli około 1-25 punktów; odpowiada to ograniczeniom Stan `0.25-25`. Prior pozwala na małe i umiarkowane efekty sezonu, ale nie pozwala im dominować nad jakością drużyn.

`log_sigma_pts ~ normal(log(15), 0.8)` opisuje resztkową losowość pojedynczego sezonu po uwzględnieniu siły drużyny, efektu sezonu i statusu beniaminka. Mediana to 15 punktów. Nieucięty orientacyjny 95% zakres to `15 * exp(±1.6)`, czyli około 3-74 punkty, ale ograniczenia Stan obcinają go do `1-35`. To uznaje, że sezony mogą być mocno losowe, ale unika absurdalnych skal reszt.

Skala logarytmiczna jest używana dla `sigma_pts`, `tau_team` i `tau_season`, bo te parametry muszą być dodatnie. Prior predictive check traktujemy wyłącznie jako sanity check: ma pokazać, czy implikowane punkty są piłkarsko możliwe, ale nie służy do strojenia priorów pod obserwowany rozkład danych.


## Dlaczego model hierarchiczny nie powinien być zbyt elastyczny?

Model drużynowy ma mało obserwacji: jedna liczba punktów na drużynę w sezonie. Jeśli każdej parze sezon–drużyna damy prawie niezależny skill, model może bardzo dobrze dopasować historię, ale gorzej prognozować przyszłość.

Dlatego finalna konstrukcja jest taka:

```text
skill = trwała jakość drużyny + wspólny efekt sezonu
```

Trwała jakość przenosi informację między sezonami. Wspólny efekt sezonu pozwala przesuwać cały sezon w górę lub w dół, a indywidualna zmienność drużyn pozostaje w Student-t residual noise.

## Backtest i finalny forecast

W podstawowym backteście trenujemy modele na sezonach do 2024/25 i przewidujemy **punkty każdej drużyny** w sezonie 2025/26. Szczegóły per model: `02_model1_static_team` (Model 1) i `03_model2_hierarchical_team` (Model 2).

Notebook `05_backtest_models_comparison` porównuje **oba modele** na tym samym backteście: błędy punktów per drużyna, suma błędów sezonu i średnia na drużynę (`sum |error| / n_teams`).

W finalnym forecastcie trenujemy na wszystkich dostępnych sezonach, włącznie z 2025/26, i przewidujemy sezon 2026/27 (`04_forecast_2627_comparison`).

Dla drużyn prognozowanego sezonu Model 1 buduje cechy procesowe na podstawie ostatniego dostępnego sezonu. Model 2 ignoruje te cechy i używa tylko `is_promoted`, posteriorowego `team_skill` oraz losowanego wspólnego `season_effect`. Drużyny, które nie grały w poprzednim sezonie Premier League, dostają `is_promoted=1`, a oba modele mają parametr `beta_promoted`, który może obniżyć ich oczekiwane punkty.

**Skill w prognozie:** Model 1 używa stałego `skill[team]` oraz cech procesowych. Model 2 bierze `team_skill[team]` z posterioru, losuje nowy wspólny `season_effect` dla prognozowanego sezonu i używa tylko `is_promoted` z cech. Drużyny bez historii w treningu dostają skill = 0; pozostała niepewność pochodzi z predykcyjnego Student-t residual noise oraz `beta_promoted` dla beniaminków.

## Diagnostyka modeli

Po dopasowaniu modelu sprawdzamy:

- `R_hat` - powinno być blisko 1.00, zwykle akceptujemy do 1.01,
- `ESS_bulk` i `ESS_tail` - im większe, tym lepiej; niskie ESS oznacza słabe mieszanie łańcuchów,
- divergences - powinno być 0,
- treedepth - nie powinien masowo dobijać do maksimum,
- E-BFMI - niskie wartości sugerują problemy z geometrią posterioru.

Diagnostyka samplera mówi, czy posterior został dobrze wysamplowany. Nie mówi jeszcze, czy forecast jest dobry. Jakość predykcji sprawdzamy osobno przez backtest: błędy punktów (`predict_team_points` vs rzeczywiste Pts) w `02_model1_static_team`, `03_model2_hierarchical_team` i porównanie obu modeli w `05_backtest_models_comparison`.

## Krótkie porównanie modeli drużynowych

`team_static`:

- prostszy,
- stabilny diagnostycznie,
- ma jeden skill drużyny dla wszystkich sezonów,
- często daje dobrą predykcję, bo mocno korzysta z trwałej jakości drużyn.

`team_hierarchical`:

- bardziej elastyczny,
- rozdziela trwały skill drużyny i wspólny efekt sezonu,
- nie używa cech procesowych, więc jest czystszym hierarchicznym baseline’em,
- wymaga regularizacji skal `tau_team`, `tau_season` i `sigma_pts`, żeby nie mylić trwałej jakości, efektów sezonu i szumu.

W praktyce model hierarchiczny nie powinien być oceniany po samej złożoności. Trzeba patrzeć na backtest. Bardziej złożony model może gorzej przewidywać, jeśli dodatkowa elastyczność łapie szum historyczny.